**Pipeline stats**

In [ ]:
import os

os.environ["AZURE_OPENAI_ENDPOINT"] = "https://<YOUR_OPENAI_RESOURCE>.services.ai.azure.com/"
os.environ["AZURE_OPENAI_API_KEY"] = "<YOUR_OPENAI_KEY>"
os.environ["AZURE_OPENAI_DEPLOYMENT"] = "gpt-4o"
os.environ["AZURE_OPENAI_API_VERSION"] = "2025-01-01-preview"
os.environ["AZURE_OPENAI_EMBEDDING_DEPLOYMENT"] = "text-embedding-3-small"
os.environ["ADLS_STORAGE_ACCOUNT"] = "<YOUR_ADLS_ACCOUNT>"
os.environ["ADLS_CONTAINER"] = "<YOUR_ADLS_CONTAINER>"
os.environ["ADLS_STORAGE_KEY"] = "<YOUR_ADLS_KEY>"
os.environ["COSMOS_GREMLIN_ENDPOINT"] = "wss://<YOUR_COSMOS_ACCOUNT>.gremlin.cosmos.azure.com:443/"
os.environ["COSMOS_GREMLIN_KEY"] = "<YOUR_COSMOS_KEY>"
os.environ["COSMOS_DATABASE"] = "email_cosmos_db"
os.environ["COSMOS_GRAPH"] = "email_cosmos_graph"
os.environ["COSMOS_NOSQL_ENDPOINT"] = ""
os.environ["COSMOS_NOSQL_KEY"] = ""
os.environ["AZURE_SEARCH_ENDPOINT"] = "https://<YOUR_SEARCH_SERVICE>.search.windows.net"
os.environ["AZURE_SEARCH_API_KEY"] = "<YOUR_SEARCH_KEY>"
os.environ["AZURE_SEARCH_INDEX_NAME"] = "kg-chunks"
os.environ["AZURE_SEARCH_ENTITY_INDEX_NAME"] = "kg-entities"
os.environ["AZURE_SEARCH_COMMUNITY_INDEX_NAME"] = "kg-communities"
os.environ["PIPELINE_MODE"] = "llm"
os.environ["PROCESS_ATTACHMENTS"] = "true"
print("Environment variables set.")

**Download code from ADLS**

In [ ]:
import os
import sys
from azure.identity import DefaultAzureCredential
from azure.storage.filedatalake import DataLakeServiceClient

service = DataLakeServiceClient(
    account_url=f"https://{os.environ['ADLS_STORAGE_ACCOUNT']}.dfs.core.windows.net",
    credential=os.environ["ADLS_STORAGE_KEY"]
)

fs = service.get_file_system_client(os.environ["ADLS_CONTAINER"])

for prefix in ["code/src", "code/config"]:
    for p in fs.get_paths(path=prefix, recursive=True):
        if not p.is_directory:
            local_path = p.name.replace("code/", "", 1)
            os.makedirs(os.path.dirname(local_path), exist_ok=True)
            with open(local_path, "wb") as f:
                file_client = fs.get_file_client(p.name)
                f.write(file_client.download_file().readall())

sys.path.insert(0, ".")
src_count = len(os.listdir("src")) if os.path.exists("src") else 0
config_count = len(os.listdir("config")) if os.path.exists("config") else 0
print(f"Downloaded: {src_count} items in src/, {config_count} items in config/")

**Download graph files from ADLS**

In [ ]:
from src.storage.adls_adapter import ADLSAdapter
import json
from collections import Counter

adls = ADLSAdapter()
MODE = os.getenv("PIPELINE_MODE", "llm")
LOCAL_GOLD = f"/tmp/pipeline/gold_{MODE}"

os.makedirs(f"{LOCAL_GOLD}/knowledge_graph", exist_ok=True)

for fname in ["nodes.json", "edges.json"]:
    data = adls.read_bytes(f"gold_{MODE}/knowledge_graph/{fname}")
    if data:
        with open(f"{LOCAL_GOLD}/knowledge_graph/{fname}", "wb") as f:
            f.write(data)
        print(f"Downloaded {fname} ({len(data)} bytes)")

catalog_data = adls.read_bytes(f"gold_{MODE}/entity_catalog.json")
if catalog_data:
    with open(f"{LOCAL_GOLD}/entity_catalog.json", "wb") as f:
        f.write(catalog_data)
    print(f"Downloaded entity_catalog.json ({len(catalog_data)} bytes)")

print("Graph files ready.")

**Knowledge Graph Stats**

In [ ]:

print("=" * 70)
print("PIPELINE STATISTICS")
print("=" * 70)

# --- Nodes ---
with open(f"{LOCAL_GOLD}/knowledge_graph/nodes.json") as f:
    nodes = json.load(f)

node_types = Counter()
for node_id, node in nodes.items():
    ntype = node.get("node_type", "UNKNOWN")
    node_types[ntype] += 1

print(f"\n--- Knowledge Graph ---")
print(f"  Total nodes:                  {len(nodes)}")
print(f"\n  Node types:")
for ntype, count in node_types.most_common():
    print(f"    {ntype:25s} {count}")

# --- Edges ---
with open(f"{LOCAL_GOLD}/knowledge_graph/edges.json") as f:
    edges_raw = json.load(f)
edges = list(edges_raw.values()) if isinstance(edges_raw, dict) else edges_raw

edge_types = Counter()
for edge in edges:
    if isinstance(edge, dict):
        edge_types[edge.get("edge_type", "UNKNOWN")] += 1

print(f"\n--- Edges ---")
print(f"  Total edges:                  {len(edges)}")
print(f"\n  Edge types:")
for etype, count in edge_types.most_common():
    print(f"    {etype:25s} {count}")

**Entity Catalog Stats**

In [ ]:
catalog_file = f"{LOCAL_GOLD}/entity_catalog.json"
if os.path.exists(catalog_file):
    with open(catalog_file) as f:
        catalog = json.load(f)

    entities = catalog.get("entities", {})
    catalog_types = Counter()
    has_aliases = 0
    total_aliases = 0
    for name, info in entities.items():
        catalog_types[info.get("type", "UNKNOWN")] += 1
        aliases = info.get("aliases", [])
        if aliases:
            has_aliases += 1
            total_aliases += len(aliases)

    print(f"\n--- Entity Catalog ---")
    print(f"  Total catalog entries:        {len(entities)}")
    print(f"  Entries with aliases:         {has_aliases}")
    print(f"  Total aliases:                {total_aliases}")
    print(f"\n  Catalog entity types:")
    for ctype, count in catalog_types.most_common():
        print(f"    {ctype:25s} {count}")

**Cosmos DB Gremlin Stats**

In [ ]:
from src.storage.cosmos_adapter import CosmosAdapter

cosmos = CosmosAdapter.from_env()

if cosmos.is_configured and cosmos.gremlin_endpoint:
    print(f"\n--- Cosmos DB Gremlin ---")

    # Node count
    try:
        result = cosmos._gremlin_query("g.V().count()")
        print(f"  Total vertices:               {result[0] if result else 'N/A'}")
    except Exception as e:
        print(f"  Vertex count failed: {e}")

    # Edge count
    try:
        result = cosmos._gremlin_query("g.E().count()")
        print(f"  Total edges:                  {result[0] if result else 'N/A'}")
    except Exception as e:
        print(f"  Edge count failed: {e}")

    # Node types
    try:
        result = cosmos._gremlin_query("g.V().groupCount().by('node_type')")
        if result:
            print(f"\n  Vertex types:")
            types = result[0] if result else {}
            for ntype, count in sorted(types.items(), key=lambda x: x[1], reverse=True):
                print(f"    {ntype:25s} {count}")
    except Exception as e:
        print(f"  Node type query failed: {e}")

    # Edge types
    # try:
    #     result = cosmos._gremlin_query("g.E().groupCount().by(label)")
    #     if result:
    #         print(f"\n  Edge types:")
    #         types = result[0] if result else {}
    #         for etype, count in sorted(types.items(), key=lambda x: x[1], reverse=True)[:15]:
    #             print(f"    {etype:25s} {count}")
    # except Exception as e:
    #     print(f"  Edge type query failed: {e}")
    # Edge types — use ADLS data instead of Gremlin (too slow on 800K+ edges)
    print(f"\n  Edge types (from ADLS graph):")
    for etype, count in edge_types.most_common():
        print(f"    {etype:25s} {count}")
else:
    print("\n  Cosmos DB Gremlin not configured")

**Azure AI Search Stats**

In [ ]:
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient

search_endpoint = os.environ["AZURE_SEARCH_ENDPOINT"]
search_key = os.environ["AZURE_SEARCH_API_KEY"]
credential = AzureKeyCredential(search_key)

index_client = SearchIndexClient(search_endpoint, credential)

print(f"\n--- Azure AI Search Indexes ---")

indexes = {
    "kg-chunks": "Email/attachment chunks",
    "kg-entities": "Entity embeddings",
    "kg-communities": "Community summaries",
    "kg-thread-summaries": "Thread summaries",
    "kg-attachment-summaries": "Attachment summaries",
}

for index_name, description in indexes.items():
    try:
        client = SearchClient(search_endpoint, index_name, credential)
        result = client.search(search_text="*", top=0, include_total_count=True)
        count = result.get_count()
        print(f"  {index_name:30s} {count:>8} documents  ({description})")
    except Exception as e:
        print(f"  {index_name:30s} ERROR: {e}")

**Build detailed reports**

In [ ]:
from collections import defaultdict

node_names = {}
node_types_map = {}
for node_id, node in nodes.items():
    node_names[node_id] = node.get("name", node_id)
    node_types_map[node_id] = node.get("node_type", "UNKNOWN")

entities_by_type = defaultdict(list)
for node_id, node in nodes.items():
    ntype = node.get("node_type", "UNKNOWN")
    entities_by_type[ntype].append({
        "node_id": node_id,
        "name": node.get("name", node_id),
        "mention_count": node.get("mention_count", 0),
        "source_chunks_count": len(node.get("source_chunks", [])),
        "properties": node.get("properties", {}),
    })

for ntype in entities_by_type:
    entities_by_type[ntype].sort(key=lambda x: x["mention_count"], reverse=True)

entity_report = {
    "total_entities": sum(len(v) for v in entities_by_type.values()),
    "types": {
        ntype: {
            "count": len(ents),
            "total_mentions": sum(e["mention_count"] for e in ents),
            "entities": ents,
        }
        for ntype, ents in sorted(entities_by_type.items(), key=lambda x: len(x[1]), reverse=True)
    },
}

rel_pairs = defaultdict(list)
for edge in edges:
    etype = edge.get("edge_type", "UNKNOWN")
    src_id = edge.get("source_id", "")
    tgt_id = edge.get("target_id", "")
    src_type = node_types_map.get(src_id, "UNKNOWN")
    tgt_type = node_types_map.get(tgt_id, "UNKNOWN")

    pair_key = f"{src_type} --[{etype}]--> {tgt_type}"
    rel_pairs[pair_key].append({
        "source": node_names.get(src_id, src_id),
        "target": node_names.get(tgt_id, tgt_id),
        "weight": edge.get("weight", 0),
        "evidence_chunks": len(edge.get("evidence_chunks", [])),
    })

relationship_report = {
    "total_edges": len(edges),
    "edge_type_counts": dict(edge_types.most_common()),
    "relationship_pairs": {
        pair: {
            "count": len(instances),
            "examples": sorted(instances, key=lambda x: x["weight"], reverse=True)[:20],
        }
        for pair, instances in sorted(rel_pairs.items(), key=lambda x: len(x[1]), reverse=True)
    },
}

print(f"Entity report: {entity_report['total_entities']} entities across {len(entity_report['types'])} types")
print(f"Relationship report: {len(relationship_report['relationship_pairs'])} pair patterns")



**Build community report**

In [ ]:
community_files = []
try:
    for p in fs.get_paths(path=f"gold_{MODE}/communities", recursive=True):
        if not p.is_directory and p.name.endswith(".json"):
            community_files.append(p.name)
except Exception as e:
    print(f"Error listing communities: {e}")

print(f"Found {len(community_files)} community files on ADLS")

communities_by_level = defaultdict(list)
for cf in community_files:
    try:
        data = fs.get_file_client(cf).download_file().readall()
        comm = json.loads(data)
        level = comm.get("level", 0)
        communities_by_level[level].append({
            "community_id": comm.get("community_id", ""),
            "level": level,
            "parent_id": comm.get("parent_id"),
            "child_ids": comm.get("child_ids", []),
            "entity_count": comm.get("entity_count", len(comm.get("node_ids", []))),
            "edge_count": comm.get("edge_count", 0),
            "density": comm.get("density", 0),
            "has_summary": bool(comm.get("summary", "").strip()),
            "summary": comm.get("summary", ""),
            "source_chunk_count": len(comm.get("source_chunk_ids", [])),
            "top_entities": [
                node_names.get(nid, nid)
                for nid in comm.get("node_ids", [])[:10]
            ],
        })
    except Exception as e:
        print(f"  Error reading {cf}: {e}")

for level in communities_by_level:
    communities_by_level[level].sort(key=lambda x: x["entity_count"], reverse=True)

community_report = {
    "total_communities": sum(len(v) for v in communities_by_level.values()),
    "levels": {},
}

print(f"\nCommunity levels:")
for level in sorted(communities_by_level.keys()):
    comms = communities_by_level[level]
    sizes = [c["entity_count"] for c in comms]
    densities = [c["density"] for c in comms]
    summarized = sum(1 for c in comms if c["has_summary"])

    level_info = {
        "count": len(comms),
        "total_entities": sum(sizes),
        "avg_entities": round(sum(sizes) / len(sizes), 1) if sizes else 0,
        "min_entities": min(sizes) if sizes else 0,
        "max_entities": max(sizes) if sizes else 0,
        "avg_density": round(sum(densities) / len(densities), 3) if densities else 0,
        "summarized": summarized,
        "communities": comms,
    }
    community_report["levels"][str(level)] = level_info

    print(f"  Level {level}: {len(comms)} communities, "
        f"avg {level_info['avg_entities']} entities "
        f"(min {level_info['min_entities']}, max {level_info['max_entities']}), "
        f"avg density {level_info['avg_density']}, "
        f"{summarized} summarized")

print(f"  Total: {community_report['total_communities']} communities")


**Write reports to ADLS**

In [ ]:
def write_json_adls(path, obj):
    data = json.dumps(obj, indent=2, ensure_ascii=False).encode("utf-8")
    fc = fs.get_file_client(path)           
    fc.upload_data(data, overwrite=True)
    print(f"  Wrote {path} ({len(data)/1024:.1f} KB)")
                                                                                                                                                            
stats_prefix = f"gold_{MODE}/stats"
                                                                                                                                                            
write_json_adls(f"{stats_prefix}/entity_report.json", entity_report)
write_json_adls(f"{stats_prefix}/relationship_report.json", relationship_report)                                                                              
write_json_adls(f"{stats_prefix}/community_report.json", community_report)
                                                                                                                                                            
summary = {
    "total_nodes": len(nodes),                                                                                                                                
    "total_edges": len(edges),                            
    "total_entities": entity_report["total_entities"],
    "total_communities": community_report["total_communities"],
    "node_type_counts": dict(node_types.most_common()),                                                                                                       
    "edge_type_counts": relationship_report["edge_type_counts"],
    "relationship_pair_counts": {                                                                                                                             
        p: info["count"] for p, info in relationship_report["relationship_pairs"].items()
    },                                                                                                                                                        
    "community_level_counts": {
        level: info["count"] for level, info in community_report["levels"].items()                                                                            
    },                                                    
    "community_level_avg_entities": {
        level: info["avg_entities"] for level, info in community_report["levels"].items()                                                                     
    },                                      
}                                                                                                                                                             
write_json_adls(f"{stats_prefix}/graph_summary.json", summary)
                                                                                                                                                            
print(f"\nDone — 4 files written to {stats_prefix}/")